# Entraînement des modèles

Notebook pour tester et entraîner les modèles.

In [1]:
# Cellule 1 — imports et chargement
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import joblib
import warnings
warnings.filterwarnings("ignore")

dataset = pd.read_csv("../data/processed/dataset_enrichi.csv")
print(f"Dataset chargé : {dataset.shape}")

Dataset chargé : (220, 114)


In [2]:
# Cellule 2 — préparation des features et de la cible
# Exclure les colonnes non numériques et la variable cible
cols_exclure = ["departement", "rendement_kg_ha", "rendement_t_ha"]
X = dataset.drop(columns=cols_exclure)
y = dataset["rendement_t_ha"]

print(f"Features X : {X.shape}")
print(f"Cible y    : {y.shape}")
print(f"\nExtrait des colonnes :")
print(X.columns.tolist()[:10])

Features X : (220, 111)
Cible y    : (220,)

Extrait des colonnes :
['annee', 'precip_01_mm', 'temp_moy_01_c', 'temp_max_01_c', 'temp_min_01_c', 'humidite_01_pct', 'rayonnement_01', 'precip_02_mm', 'temp_moy_02_c', 'temp_max_02_c']


In [3]:
# Cellule 3 — séparation train/test stratifiée par département
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Train : {X_train.shape} — {len(y_train)} observations")
print(f"Test  : {X_test.shape} — {len(y_test)} observations")
print(f"\nDistribution du rendement :")
print(f"Train — moyenne : {y_train.mean():.3f} t/ha, std : {y_train.std():.3f}")
print(f"Test  — moyenne : {y_test.mean():.3f} t/ha, std : {y_test.std():.3f}")

Train : (176, 111) — 176 observations
Test  : (44, 111) — 44 observations

Distribution du rendement :
Train — moyenne : 1.224 t/ha, std : 0.322
Test  — moyenne : 1.221 t/ha, std : 0.332


In [4]:
# Cellule 4 — fonction d'évaluation commune aux trois modèles
def evaluer_modele(nom, modele, X_train, X_test, y_train, y_test):
    # Entraînement
    modele.fit(X_train, y_train)
    
    # Prédictions
    y_pred_train = modele.predict(X_train)
    y_pred_test  = modele.predict(X_test)
    
    # Métriques
    resultats = {
        "Modele":        nom,
        "RMSE_train":    np.sqrt(mean_squared_error(y_train, y_pred_train)),
        "RMSE_test":     np.sqrt(mean_squared_error(y_test,  y_pred_test)),
        "R2_train":      r2_score(y_train, y_pred_train),
        "R2_test":       r2_score(y_test,  y_pred_test),
        "MAE_test":      mean_absolute_error(y_test, y_pred_test),
    }
    
    print(f"\n{'='*40}")
    print(f"  {nom}")
    print(f"{'='*40}")
    print(f"  RMSE train : {resultats['RMSE_train']:.4f} t/ha")
    print(f"  RMSE test  : {resultats['RMSE_test']:.4f} t/ha")
    print(f"  R²   train : {resultats['R2_train']:.4f}")
    print(f"  R²   test  : {resultats['R2_test']:.4f}")
    print(f"  MAE  test  : {resultats['MAE_test']:.4f} t/ha")
    
    return resultats, y_pred_test

tous_resultats = []

In [5]:
# Cellule 5 — Modèle 1 : Régression linéaire multiple (baseline)
modele_lr = LinearRegression()
resultats_lr, y_pred_lr = evaluer_modele(
    "Régression Linéaire",
    modele_lr,
    X_train, X_test, y_train, y_test
)
tous_resultats.append(resultats_lr)


  Régression Linéaire
  RMSE train : 0.1369 t/ha
  RMSE test  : 0.3188 t/ha
  R²   train : 0.8183
  R²   test  : 0.0572
  MAE  test  : 0.2584 t/ha
